In [ ]:
!pip install scipy

In [ ]:
import numpy as np
from scipy.stats import norm

# ============================================================
# BLACK-SCHOLES-MERTON OPTION PRICER
# ============================================================

def bsm_price(S, K, T, r, sigma, option_type="call"):
    """
    Calculate BSM option price.
    
    Parameters:
    S     : Current stock price
    K     : Strike price
    T     : Time to expiry (in years)
    r     : Risk-free rate (annual, as decimal e.g. 0.05 = 5%)
    sigma : Volatility (annual, as decimal e.g. 0.30 = 30%)
    option_type : "call" or "put"
    
    Returns:
    Option price
    """
    
    # d1 and d2 are the core BSM intermediate values
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if option_type == "call":
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif option_type == "put":
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    
    return price

# ============================================================
# TEST IT ON RKLB
# ============================================================

S = 105.47      # current RKLB price
K = 110.00      # strike price (slightly out of the money)
T = 0.25        # 3 months to expiry
r = 0.043       # US 10-year treasury yield (~4.3%)
sigma = 0.8782  # RKLB annualised vol from our Monte Carlo

call_price = bsm_price(S, K, T, r, sigma, "call")
put_price  = bsm_price(S, K, T, r, sigma, "put")

print("=" * 45)
print("  RKLB BSM Option Prices")
print("=" * 45)
print(f"  Stock price (S):      ${S:.2f}")
print(f"  Strike price (K):     ${K:.2f}")
print(f"  Time to expiry (T):   {T*12:.0f} months")
print(f"  Risk-free rate (r):   {r:.1%}")
print(f"  Volatility (σ):       {sigma:.1%}")
print("=" * 45)
print(f"  Call price:           ${call_price:.2f}")
print(f"  Put price:            ${put_price:.2f}")
print("=" * 45)

# Quick sanity check using Put-Call Parity
# C - P should equal S - K*e^(-rT)
parity = call_price - put_price
theoretical = S - K * np.exp(-r * T)
print(f"\n  Put-Call Parity check:")
print(f"  C - P:                ${parity:.2f}")
print(f"  S - Ke^(-rT):         ${theoretical:.2f}")
print(f"  Parity holds:         {abs(parity - theoretical) < 0.01}")

In [ ]:
def bsm_greeks(S, K, T, r, sigma, option_type="call"):
    """
    Calculate BSM option Greeks.
    
    Returns dictionary of all Greeks for call or put.
    """
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    # --- Gamma and Vega are same for calls and puts ---
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega  = S * norm.pdf(d1) * np.sqrt(T) / 100  # per 1% change in vol

    if option_type == "call":
        delta = norm.cdf(d1)
        theta = (-(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
                 - r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
        rho   = K * T * np.exp(-r * T) * norm.cdf(d2) / 100

    elif option_type == "put":
        delta = norm.cdf(d1) - 1
        theta = (-(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
                 + r * K * np.exp(-r * T) * norm.cdf(-d2)) / 365
        rho   = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100

    return {
        "delta": delta,
        "gamma": gamma,
        "theta": theta,
        "vega":  vega,
        "rho":   rho
    }

# ============================================================
# CALCULATE AND DISPLAY GREEKS FOR RKLB
# ============================================================

call_greeks = bsm_greeks(S, K, T, r, sigma, "call")
put_greeks  = bsm_greeks(S, K, T, r, sigma, "put")

print("=" * 55)
print("  RKLB Option Greeks")
print("=" * 55)
print(f"  {'Greek':<10} {'Call':>15} {'Put':>15}")
print("-" * 55)
print(f"  {'Delta':.<10} {call_greeks['delta']:>15.4f} {put_greeks['delta']:>15.4f}")
print(f"  {'Gamma':.<10} {call_greeks['gamma']:>15.4f} {put_greeks['gamma']:>15.4f}")
print(f"  {'Theta':.<10} {call_greeks['theta']:>15.4f} {put_greeks['theta']:>15.4f}")
print(f"  {'Vega':.<10} {call_greeks['vega']:>15.4f} {put_greeks['vega']:>15.4f}")
print(f"  {'Rho':.<10} {call_greeks['rho']:>15.4f} {put_greeks['rho']:>15.4f}")
print("=" * 55)

print(f"""
  What this means for your RKLB options:

  Call Delta  {call_greeks['delta']:.4f} — for every $1 RKLB rises, 
              your call gains ${call_greeks['delta']:.2f}

  Put Delta   {put_greeks['delta']:.4f} — for every $1 RKLB falls,
              your put gains ${abs(put_greeks['delta']):.2f}

  Theta       ${call_greeks['theta']:.4f} — your option loses 
              ${abs(call_greeks['theta']):.2f} per day just from time decay

  Vega        {call_greeks['vega']:.4f} — for every 1% rise in vol,
              your option gains ${call_greeks['vega']:.2f}
""")

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# ============================================================
# DEFAULT VALUES
# ============================================================

DEFAULTS = {
    "S": 105.47,
    "K": 110.00,
    "T": 3,
    "r": 4.3,
    "sigma": 87.82
}

# ============================================================
# PLOT FUNCTION
# ============================================================

def update_pricer(S, K, T_months, r_pct, sigma_pct):
    T = T_months / 12
    r = r_pct / 100
    sigma = sigma_pct / 100

    call = bsm_price(S, K, T, r, sigma, "call")
    put  = bsm_price(S, K, T, r, sigma, "put")
    cg   = bsm_greeks(S, K, T, r, sigma, "call")
    pg   = bsm_greeks(S, K, T, r, sigma, "put")

    fig = plt.figure(figsize=(13, 7))
    fig.patch.set_facecolor("#0d0d1a")
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.4)

    moneyness = S / K
    if moneyness > 1.05:
        status, colour = "IN THE MONEY", "#28a745"
    elif moneyness < 0.95:
        status, colour = "OUT OF THE MONEY", "#dc3545"
    else:
        status, colour = "AT THE MONEY", "#fd7e14"

    fig.suptitle(
        f"RKLB BSM Pricer  |  S=${S:.2f}  K=${K:.2f}  "
        f"T={T_months}m  r={r_pct}%  σ={sigma_pct}%  |  {status}",
        color="white", fontsize=11, y=0.98
    )

    def make_ax(pos):
        ax = fig.add_subplot(pos)
        ax.set_facecolor("#1a1a2e")
        ax.tick_params(colors="white", labelsize=8)
        for spine in ax.spines.values():
            spine.set_edgecolor("#444")
        return ax

    ax1 = make_ax(gs[0, 0])
    bars = ax1.bar(["Call", "Put"], [call, put],
                   color=[colour, "#4a9edd"], width=0.4, edgecolor="none")
    ax1.set_title("Option Prices", color="white", fontsize=9)
    ax1.set_ylabel("Price ($)", color="white", fontsize=8)
    for bar, val in zip(bars, [call, put]):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f"${val:.2f}", ha="center", color="white", fontsize=10, fontweight="bold")
    ax1.set_ylim(0, max(call, put) * 1.3)

    ax2 = make_ax(gs[0, 1])
    ax2.barh(["Put Delta", "Call Delta"],
             [pg["delta"], cg["delta"]],
             color=["#dc3545", colour], edgecolor="none")
    ax2.set_title("Delta", color="white", fontsize=9)
    ax2.axvline(0, color="white", linewidth=0.5)
    ax2.set_xlim(-1, 1)
    for i, val in enumerate([pg["delta"], cg["delta"]]):
        ax2.text(val + (0.03 if val > 0 else -0.03), i,
                 f"{val:.3f}", va="center", color="white", fontsize=9)

    ax3 = make_ax(gs[0, 2])
    ax3.bar(["Vega", "Gamma"], [cg["vega"], cg["gamma"]],
            color=["#9b59b6", "#f39c12"], edgecolor="none", width=0.4)
    ax3.set_title("Vega & Gamma (Call)", color="white", fontsize=9)
    ax3.tick_params(colors="white")

    ax4 = make_ax(gs[1, 0])
    S_range = np.linspace(S * 0.5, S * 1.5, 200)
    call_curve = [bsm_price(s, K, T, r, sigma, "call") for s in S_range]
    put_curve  = [bsm_price(s, K, T, r, sigma, "put")  for s in S_range]
    ax4.plot(S_range, call_curve, color=colour, linewidth=1.5, label="Call")
    ax4.plot(S_range, put_curve,  color="#4a9edd", linewidth=1.5, label="Put")
    ax4.axvline(S, color="white", linewidth=1, linestyle="--", alpha=0.5)
    ax4.axvline(K, color="yellow", linewidth=1, linestyle=":", alpha=0.5)
    ax4.set_title("Price vs Stock Price", color="white", fontsize=9)
    ax4.set_xlabel("Stock Price ($)", color="white", fontsize=8)
    ax4.set_ylabel("Option Price ($)", color="white", fontsize=8)
    ax4.legend(fontsize=7, facecolor="#1a1a2e", labelcolor="white")

    ax5 = make_ax(gs[1, 1])
    T_range = np.linspace(0.01, T, 200)
    call_theta = [bsm_price(S, K, t, r, sigma, "call") for t in T_range]
    ax5.plot(T_range * 12, call_theta, color=colour, linewidth=1.5)
    ax5.axvline(T_months, color="white", linewidth=1, linestyle="--", alpha=0.5)
    ax5.set_title("Call Price Decay Over Time", color="white", fontsize=9)
    ax5.set_xlabel("Months to Expiry", color="white", fontsize=8)
    ax5.set_ylabel("Call Price ($)", color="white", fontsize=8)

    ax6 = make_ax(gs[1, 2])
    ax6.axis("off")
    table_data = [
        ["Greek", "Call", "Put"],
        ["Delta",  f"{cg['delta']:.4f}",  f"{pg['delta']:.4f}"],
        ["Gamma",  f"{cg['gamma']:.4f}",  f"{pg['gamma']:.4f}"],
        ["Theta",  f"{cg['theta']:.4f}",  f"{pg['theta']:.4f}"],
        ["Vega",   f"{cg['vega']:.4f}",   f"{pg['vega']:.4f}"],
        ["Rho",    f"{cg['rho']:.4f}",    f"{pg['rho']:.4f}"],
    ]
    table = ax6.table(cellText=table_data[1:], colLabels=table_data[0],
                      cellLoc="center", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    for (row, col), cell in table.get_celld().items():
        cell.set_facecolor("#1a1a2e" if row % 2 == 0 else "#0d0d1a")
        cell.set_text_props(color="white")
        cell.set_edgecolor("#444")
    ax6.set_title("Greeks Summary", color="white", fontsize=9)

    plt.show()

# ============================================================
# SLIDERS — finer increments
# ============================================================

style  = {"description_width": "140px"}
layout = widgets.Layout(width="500px")

S_slider     = widgets.FloatSlider(value=DEFAULTS["S"],    min=50.0,  max=200.0, step=0.01,
                                    description="Stock Price (S):",    style=style, layout=layout)
K_slider     = widgets.FloatSlider(value=DEFAULTS["K"],    min=50.0,  max=200.0, step=0.01,
                                    description="Strike Price (K):",   style=style, layout=layout)
T_slider     = widgets.FloatSlider(value=DEFAULTS["T"],    min=0.5,   max=24.0,  step=0.5,
                                    description="Expiry (months):",    style=style, layout=layout)
r_slider     = widgets.FloatSlider(value=DEFAULTS["r"],    min=0.0,   max=10.0,  step=0.05,
                                    description="Risk-free Rate (%):", style=style, layout=layout)
sigma_slider = widgets.FloatSlider(value=DEFAULTS["sigma"],min=10.0,  max=200.0, step=0.1,
                                    description="Volatility (%):",     style=style, layout=layout)

# ============================================================
# RESET BUTTON
# ============================================================

reset_button = widgets.Button(
    description="Reset to Defaults",
    button_style="warning",
    icon="refresh",
    layout=widgets.Layout(width="200px", margin="10px 0px")
)

def on_reset(b):
    S_slider.value     = DEFAULTS["S"]
    K_slider.value     = DEFAULTS["K"]
    T_slider.value     = DEFAULTS["T"]
    r_slider.value     = DEFAULTS["r"]
    sigma_slider.value = DEFAULTS["sigma"]

reset_button.on_click(on_reset)

# ============================================================
# DISPLAY
# ============================================================

ui = widgets.VBox([S_slider, K_slider, T_slider, r_slider, sigma_slider, reset_button])
out = widgets.interactive_output(update_pricer, {
    "S": S_slider, "K": K_slider, "T_months": T_slider,
    "r_pct": r_slider, "sigma_pct": sigma_slider
})

display(ui, out)

In [ ]:
import plotly.graph_objects as go
import numpy as np

# ============================================================
# VOLATILITY SURFACE
# ============================================================

# --- Ranges ---
strikes    = np.linspace(60, 180, 40)       # strike prices
expiries   = np.linspace(0.08, 2.0, 40)     # expiry in years (1 month to 2 years)

# --- Build price surface ---
call_surface = np.zeros((len(expiries), len(strikes)))

for i, T in enumerate(expiries):
    for j, K in enumerate(strikes):
        call_surface[i, j] = bsm_price(S=105.47, K=K, T=T,
                                        r=0.043, sigma=0.8782, option_type="call")

# --- Moneyness labels on X axis ---
moneyness = strikes / 105.47

# ============================================================
# PLOT
# ============================================================

fig = go.Figure(data=[go.Surface(
    x=strikes,
    y=expiries * 12,      # convert to months for readability
    z=call_surface,
    colorscale=[
        [0.0,  "rgb(20,  20,  80)"],
        [0.25, "rgb(50,  100, 200)"],
        [0.5,  "rgb(30,  180, 100)"],
        [0.75, "rgb(220, 180, 30)"],
        [1.0,  "rgb(220, 50,  50)"],
    ],
    opacity=0.9,
    contours=dict(
        z=dict(show=True, color="rgba(255,255,255,0.15)", width=1)
    ),
    hovertemplate=(
        "Strike: $%{x:.2f}<br>"
        "Expiry: %{y:.1f} months<br>"
        "Call Price: $%{z:.2f}<extra></extra>"
    ),
    colorbar=dict(
        title="Call Price ($)",
        tickfont=dict(color="white")
    )
)])

# --- Current stock price plane ---
fig.add_trace(go.Surface(
    x=[105.47, 105.47],
    y=[0, 24],
    z=[[0, 0], [80, 80]],
    opacity=0.15,
    colorscale=[[0, "yellow"], [1, "yellow"]],
    showscale=False,
    hoverinfo="skip",
    name="Current price"
))

fig.update_layout(
    title=dict(
        text="RKLB Call Option Price Surface — Strike vs Expiry",
        x=0.5,
        font=dict(color="white", size=15)
    ),
    scene=dict(
        xaxis=dict(
            title="Strike Price ($)",
            backgroundcolor="#0d0d1a",
            gridcolor="rgba(255,255,255,0.1)",
            tickfont=dict(color="white")
        ),
        yaxis=dict(
            title="Expiry (months)",
            backgroundcolor="#0d0d1a",
            gridcolor="rgba(255,255,255,0.1)",
            tickfont=dict(color="white")
        ),
        zaxis=dict(
            title="Call Price ($)",
            backgroundcolor="#0d0d1a",
            gridcolor="rgba(255,255,255,0.1)",
            tickfont=dict(color="white")
        ),
        bgcolor="#0d0d1a",
        camera=dict(
            eye=dict(x=1.7, y=-1.7, z=0.8)
        )
    ),
    paper_bgcolor="#0d0d1a",
    height=700,
    margin=dict(l=0, r=0, t=60, b=0)
)

fig.show()

print(f"\n  Reading the surface:")
print(f"  — Deep in the money (low strike) = expensive calls, shown in red/yellow")
print(f"  — Out of the money (high strike) = cheap calls, shown in blue/purple")
print(f"  — Longer expiry = more expensive across all strikes (more time value)")
print(f"  — The yellow plane marks RKLB's current price of $105.47")